In [2]:
import pandas as pd

data = {
    "Product": [
        "Laptop", "Mouse", "Keyboard", "Monitor", "Printer",
        "Tablet", "Headphones", "Camera", "Smartphone", "Speaker",
        "Laptop", "Mouse", "Keyboard", "Monitor", "Printer"
    ],
    "Category": [
        "Electronics", "Electronics", "Electronics", "Electronics", "Electronics",
        "Electronics", "Accessories", "Accessories", "Electronics", "Accessories",
        "Electronics", "Electronics", "Electronics", "Electronics", "Electronics"
    ],
    "Region": [
        "North", "North", "South", "South", "East",
        "East", "North", "South", "West", "West",
        "South", "East", "North", "West", "North"
    ],
    "Month": [
        "Jan", "Jan", "Jan", "Jan", "Jan",
        "Feb", "Feb", "Feb", "Feb", "Feb",
        "Mar", "Mar", "Mar", "Mar", "Mar"
    ],
    "Sales": [
        50000, 8000, 12000, 20000, 15000,
        30000, 7000, 25000, 60000, 9000,
        52000, 8500, 13000, 22000, 16000
    ]
}

df = pd.DataFrame(data)

# Month is Jan, Feb, Mar (strings)… so sorting may be wrong? 
# Yes 100% correct. So we need to set the order of the months.
month_order = ['Jan', 'Feb', 'Mar']

df['Month'] = pd.Categorical(df['Month'],categories=month_order,ordered=True)

In [3]:
#Sort by Category and Month properly.
df_sorted = df.sort_values(['Category', 'Month'])
print(df_sorted[['Category', 'Month', 'Sales']])

       Category Month  Sales
6   Accessories   Feb   7000
7   Accessories   Feb  25000
9   Accessories   Feb   9000
0   Electronics   Jan  50000
1   Electronics   Jan   8000
2   Electronics   Jan  12000
3   Electronics   Jan  20000
4   Electronics   Jan  15000
5   Electronics   Feb  30000
8   Electronics   Feb  60000
10  Electronics   Mar  52000
11  Electronics   Mar   8500
12  Electronics   Mar  13000
13  Electronics   Mar  22000
14  Electronics   Mar  16000


In [4]:
#Cumulative_Sales per Category
df_sorted['Cumulative_Sales'] = df_sorted.groupby('Category')['Sales'].cumsum()
print(df_sorted[['Category', 'Month', 'Sales', 'Cumulative_Sales']])

       Category Month  Sales  Cumulative_Sales
6   Accessories   Feb   7000              7000
7   Accessories   Feb  25000             32000
9   Accessories   Feb   9000             41000
0   Electronics   Jan  50000             50000
1   Electronics   Jan   8000             58000
2   Electronics   Jan  12000             70000
3   Electronics   Jan  20000             90000
4   Electronics   Jan  15000            105000
5   Electronics   Feb  30000            135000
8   Electronics   Feb  60000            195000
10  Electronics   Mar  52000            247000
11  Electronics   Mar   8500            255500
12  Electronics   Mar  13000            268500
13  Electronics   Mar  22000            290500
14  Electronics   Mar  16000            306500


In [5]:
# Previous_Sales per Category
df_sorted['Previous_Sales'] = df_sorted.groupby('Category')['Sales'].shift(1)
print(df_sorted[['Category', 'Month', 'Sales', 'Previous_Sales']])

       Category Month  Sales  Previous_Sales
6   Accessories   Feb   7000             NaN
7   Accessories   Feb  25000          7000.0
9   Accessories   Feb   9000         25000.0
0   Electronics   Jan  50000             NaN
1   Electronics   Jan   8000         50000.0
2   Electronics   Jan  12000          8000.0
3   Electronics   Jan  20000         12000.0
4   Electronics   Jan  15000         20000.0
5   Electronics   Feb  30000         15000.0
8   Electronics   Feb  60000         30000.0
10  Electronics   Mar  52000         60000.0
11  Electronics   Mar   8500         52000.0
12  Electronics   Mar  13000          8500.0
13  Electronics   Mar  22000         13000.0
14  Electronics   Mar  16000         22000.0


In [7]:
# MoM_Growth% (rounded to 2 decimals)
df_sorted['MoM_Growth_%'] = (df_sorted.groupby('Category')['Sales'].pct_change() * 100).round(2)
print(df_sorted[['Category', 'Month', 'Sales', 'MoM_Growth_%']])

# Identify rows where: MoM growth > 20%
high_growth_rows = df_sorted[df_sorted['MoM_Growth_%'] > 20]
print("\nRows with MoM growth > 20%:")
print(high_growth_rows[['Category', 'Month', 'Sales', 'MoM_Growth_%']])

       Category Month  Sales  MoM_Growth_%
6   Accessories   Feb   7000           NaN
7   Accessories   Feb  25000        257.14
9   Accessories   Feb   9000        -64.00
0   Electronics   Jan  50000           NaN
1   Electronics   Jan   8000        -84.00
2   Electronics   Jan  12000         50.00
3   Electronics   Jan  20000         66.67
4   Electronics   Jan  15000        -25.00
5   Electronics   Feb  30000        100.00
8   Electronics   Feb  60000        100.00
10  Electronics   Mar  52000        -13.33
11  Electronics   Mar   8500        -83.65
12  Electronics   Mar  13000         52.94
13  Electronics   Mar  22000         69.23
14  Electronics   Mar  16000        -27.27

Rows with MoM growth > 20%:
       Category Month  Sales  MoM_Growth_%
7   Accessories   Feb  25000        257.14
2   Electronics   Jan  12000         50.00
3   Electronics   Jan  20000         66.67
5   Electronics   Feb  30000        100.00
8   Electronics   Feb  60000        100.00
12  Electronics   Mar  13

In [21]:
# Previous MoM Growth % (rounded to 2 decimals)
df_sorted['Previous_MoM_Growth_%'] = (df_sorted.groupby('Category')['MoM_Growth_%'].shift(1))

df_sorted['Growth_Acceleration'] = (df_sorted['MoM_Growth_%'] - df_sorted['Previous_MoM_Growth_%'])

In [23]:
momentum_rows = df_sorted[(df_sorted['MoM_Growth_%'] > 0) & (df_sorted['Growth_Acceleration'] > 0)]

print(momentum_rows[['Category', 'Month', 'MoM_Growth_%','Previous_MoM_Growth_%','Growth_Acceleration']])



       Category Month  MoM_Growth_%  Previous_MoM_Growth_%  \
2   Electronics   Jan         50.00                 -84.00   
3   Electronics   Jan         66.67                  50.00   
5   Electronics   Feb        100.00                 -25.00   
12  Electronics   Mar         52.94                 -83.65   
13  Electronics   Mar         69.23                  52.94   

    Growth_Acceleration  
2                134.00  
3                 16.67  
5                125.00  
12               136.59  
13                16.29  
